# Notebook 6 — Deployment Demo
## FastAPI Local Deployment — Network Anomaly Detection

In [ ]:
import subprocess
import sys

# Check if uvicorn is available
result = subprocess.run([sys.executable, '-m', 'uvicorn', '--version'],
                        capture_output=True, text=True)
print(result.stdout or result.stderr)

## Starting the Server

Open a terminal and run:
```bash
cd src/
uvicorn app:app --host 0.0.0.0 --port 8000 --reload
```

Or from the project root:
```bash
python -m uvicorn src.app:app --host 0.0.0.0 --port 8000
```

Interactive API docs available at: **http://localhost:8000/docs**

In [ ]:
import requests
import json

BASE_URL = 'http://localhost:8000'
HEADERS  = {'X-From': 'capstone-demo-notebook', 'Content-Type': 'application/json'}

# Health check
try:
    resp = requests.get(f'{BASE_URL}/health', headers=HEADERS, timeout=3)
    print('Health:', resp.json())
except requests.ConnectionError:
    print('Server not running. Start with: uvicorn src.app:app --port 8000')

In [ ]:
# Single prediction — normal traffic example
normal_record = {
    'duration': 0, 'protocol_type': 'tcp', 'service': 'http', 'flag': 'SF',
    'src_bytes': 215, 'dst_bytes': 45076, 'land': 0, 'wrong_fragment': 0,
    'urgent': 0, 'hot': 0, 'num_failed_logins': 0, 'logged_in': 1,
    'num_compromised': 0, 'root_shell': 0, 'su_attempted': 0, 'num_root': 0,
    'num_file_creations': 0, 'num_shells': 0, 'num_access_files': 0,
    'num_outbound_cmds': 0, 'is_host_login': 0, 'is_guest_login': 0,
    'count': 2, 'srv_count': 2, 'serror_rate': 0.0, 'srv_serror_rate': 0.0,
    'rerror_rate': 0.0, 'srv_rerror_rate': 0.0, 'same_srv_rate': 1.0,
    'diff_srv_rate': 0.0, 'srv_diff_host_rate': 0.0, 'dst_host_count': 255,
    'dst_host_srv_count': 255, 'dst_host_same_srv_rate': 1.0,
    'dst_host_diff_srv_rate': 0.0, 'dst_host_same_src_port_rate': 0.01,
    'dst_host_srv_diff_host_rate': 0.0, 'dst_host_serror_rate': 0.0,
    'dst_host_srv_serror_rate': 0.0, 'dst_host_rerror_rate': 0.0,
    'dst_host_srv_rerror_rate': 0.0
}

try:
    resp = requests.post(f'{BASE_URL}/predict', headers=HEADERS, json=normal_record)
    print('Normal traffic prediction:', resp.json())
except requests.ConnectionError:
    print('Server not running.')

In [ ]:
# Single prediction — attack traffic example (SYN flood signature)
attack_record = {
    'duration': 0, 'protocol_type': 'tcp', 'service': 'http', 'flag': 'S0',
    'src_bytes': 0, 'dst_bytes': 0, 'land': 0, 'wrong_fragment': 0,
    'urgent': 0, 'hot': 0, 'num_failed_logins': 0, 'logged_in': 0,
    'num_compromised': 0, 'root_shell': 0, 'su_attempted': 0, 'num_root': 0,
    'num_file_creations': 0, 'num_shells': 0, 'num_access_files': 0,
    'num_outbound_cmds': 0, 'is_host_login': 0, 'is_guest_login': 0,
    'count': 511, 'srv_count': 511, 'serror_rate': 1.0, 'srv_serror_rate': 1.0,
    'rerror_rate': 0.0, 'srv_rerror_rate': 0.0, 'same_srv_rate': 1.0,
    'diff_srv_rate': 0.0, 'srv_diff_host_rate': 0.0, 'dst_host_count': 255,
    'dst_host_srv_count': 255, 'dst_host_same_srv_rate': 1.0,
    'dst_host_diff_srv_rate': 0.0, 'dst_host_same_src_port_rate': 1.0,
    'dst_host_srv_diff_host_rate': 0.0, 'dst_host_serror_rate': 1.0,
    'dst_host_srv_serror_rate': 1.0, 'dst_host_rerror_rate': 0.0,
    'dst_host_srv_rerror_rate': 0.0
}

try:
    resp = requests.post(f'{BASE_URL}/predict', headers=HEADERS, json=attack_record)
    print('Attack traffic prediction:', resp.json())
except requests.ConnectionError:
    print('Server not running.')

In [ ]:
# Batch prediction
batch_payload = {'records': [normal_record, attack_record, normal_record]}

try:
    resp = requests.post(f'{BASE_URL}/predict/batch', headers=HEADERS, json=batch_payload)
    result = resp.json()
    print(f'Batch predictions ({result["total"]} records):')
    for i, pred in enumerate(result['predictions']):
        print(f'  Record {i+1}: {pred["label"]:8s}  prob={pred["probability"]:.4f}')
except requests.ConnectionError:
    print('Server not running.')

## API Documentation

Interactive Swagger UI: http://localhost:8000/docs  
ReDoc: http://localhost:8000/redoc

### Endpoint Summary

| Method | Endpoint | Description |
|--------|----------|-------------|
| GET | `/health` | Liveness check |
| POST | `/predict` | Single record prediction |
| POST | `/predict/batch` | Batch prediction (up to 1000 records) |

### Request Headers
- `X-From` (required): Caller identifier for traceability
- `Content-Type: application/json`

### Response Format
```json
{
  "prediction": 1,
  "label": "attack",
  "probability": 0.9987
}
```